In [21]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\soham\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\soham\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\soham\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\soham\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

Import Core Packages

In [22]:
import pandas as pd
import spacy
import re
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

In [23]:
nlp = spacy.load("en_core_web_sm")

Sample

In [24]:
text ="Don't blink! The data scientists are buying new computers for their user studies."
# Initialize tools
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
nltk_stopwords = set(stopwords.words("english"))
# Pure Python stop words set (no NLTK/spaCy dependency)
pure_python_stopwords = {"don't", "do", "n't", "the", "are", "for", "their", "a", "an", "is", "in", "it", "of", "to", "on"}

In [25]:
# ==========================================
# 1. NLTK PROCESSING PIPELINE
# ==========================================
nltk_tokens = word_tokenize(text)
# Tokenization
nltk_processed = []
for token in nltk_tokens:
    # Stop-word removal check (lowercased token check + punctuation filtering)
    is_stop = token.lower() in nltk_stopwords or not token.isalnum()
    # Stemming & Lemmatization (Pure string-based processing without POS context)
    stem = stemmer.stem(token)
    lemma = lemmatizer.lemmatize(token.lower())
    nltk_processed.append({
        "NLTK Token": token,
        "NLTK StopWords": "Yes" if is_stop else "No",
        "NLTK Stem": stem,
        "NLTK Lemma": lemma
    })

In [26]:
# ==========================================
# 2. SPACY PROCESSING PIPELINE
# ==========================================
doc = nlp(text)
# Tokenization happens automatically here
spacy_processed = []
for token in doc:
    if token.is_space:
        continue
    # spaCy natively tracks stop words and punctuation
    # flags
    is_stop = token.is_stop or token.is_punct
    spacy_processed.append({
        "spaCy Token": token.text,
        "spaCy StopWords": "Yes" if is_stop else "No",
        "spaCy Stem": "N/A", # Stemming is not supported natively by design
        "spaCy Lemma": token.lemma_
    })

In [31]:
# ==========================================
# 3. PURE PYTHON PROCESSING PIPELINE (No NLTK / spaCy)
# ==========================================
# Regex tokenization: matches words (including contractions like don't) and punctuation
pure_tokens = re.findall(r"\b\w+(?:'\w+)?\b|[^\w\s]", text)
pure_processed = []
for token in pure_tokens:
    # Pure Python stop-word & punctuation check
    is_punct = not token.isalnum()
    is_stop = token.lower() in pure_python_stopwords or is_punct
    
    # Basic Pure Python normalization (lowercasing)
    lemma_pure = token.lower()
    # Naive rule-based suffix stripping demonstration (Pure Python stem)
    stem_pure = token.lower()
    if stem_pure.endswith('s') and len(stem_pure) > 3:
        stem_pure = stem_pure[:-1]
    elif stem_pure.endswith('ing') and len(stem_pure) > 4:
        stem_pure = stem_pure[:-3]
        
    pure_processed.append({
        "Pure Token": token,
        "Pure StopWords": "Yes" if is_stop else "No",
        "Pure Stem": stem_pure if not is_punct else token,
        "Pure Lemma": lemma_pure if not is_punct else token
    })


In [32]:
# Convert processed lists to Pandas DataFrames
df_nltk = pd.DataFrame(nltk_processed)
df_spacy = pd.DataFrame(spacy_processed)
df_pure = pd.DataFrame(pure_processed)
# Concatenate the DataFrames side-by-side
df_comparison = pd.concat([df_nltk, df_spacy, df_pure], axis=1).fillna("-")
# Render the comparison sheet in your Colab cell
display(df_comparison)

,NLTK Token,NLTK StopWords,NLTK Stem,NLTK Lemma,spaCy Token,spaCy StopWords,spaCy Stem,spaCy Lemma,Pure Token,Pure StopWords,Pure Stem,Pure Lemma
0,Do,Yes,do,do,Do,Yes,N/A,do,Don't,Yes,Don't,Don't
1,n't,Yes,n't,n't,n't,Yes,N/A,not,blink,No,blink,blink
2,blink,No,blink,blink,blink,No,N/A,blink,!,Yes,!,!
3,!,Yes,!,!,!,Yes,N/A,!,The,Yes,the,the
4,The,Yes,the,the,The,Yes,N/A,the,data,No,data,data
5,data,No,data,data,data,No,N/A,data,scientists,No,scientist,scientists
6,scientists,No,scientist,scientist,scientists,No,N/A,scientist,are,Yes,are,are
7,are,Yes,are,are,are,Yes,N/A,be,buying,No,buy,buying
8,buying,No,buy,buying,buying,No,N/A,buy,new,No,new,new
9,new,No,new,new,new,No,N/A,new,computers,No,computer,computers


In [33]:
df_pure = pd.DataFrame(pure_processed)
df_comparison = pd.concat([df_pure], axis=1).fillna("-")
# Render the comparison sheet in your Colab cell
display(df_comparison)

,Pure Token,Pure StopWords,Pure Stem,Pure Lemma
0,Don't,Yes,Don't,Don't
1,blink,No,blink,blink
2,!,Yes,!,!
3,The,Yes,the,the
4,data,No,data,data
5,scientists,No,scientist,scientists
6,are,Yes,are,are
7,buying,No,buy,buying
8,new,No,new,new
9,computers,No,computer,computers
